# Phase 5: YOLOv8n classification
Run on a Colab GPU. Reuses the exact shared manifests and Phase 4 runtime-local dataset cache. Best checkpoint selection and early stopping use validation macro F1. Default experiment uses 25 epochs, batch size 32, 224-pixel inputs, AdamW, and seed 42.

In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics>=8.3,<9'])

In [ ]:
from pathlib import Path, PurePosixPath
import json
import random
import sys
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models


HEAD_EPOCHS = 6
FINETUNE_EPOCHS = 10
BATCH_SIZE = 32

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    project_root = Path('/content/drive/MyDrive/IML_RealWaste')
else:
    project_root = Path.cwd()
    if project_root.name == 'notebooks':
        project_root = project_root.parent

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device, torch.cuda.get_device_name() if device.type == 'cuda' else '')

## Validate the dataset and preserve the shared split

In [ ]:
data_dir = project_root / 'data'
metadata = json.loads((data_dir / 'dataset_meta.json').read_text(encoding='utf-8'))
classes = metadata['classes']
label_ids = list(range(len(classes)))
seed = metadata['seed']
frames = {split: pd.read_csv(data_dir / f'{split}.csv') for split in ('train', 'val', 'test')}

# Colab reads images from its local disk to avoid repeated Drive filesystem access.
# Keep manifests and experiment outputs in Drive.
if 'google.colab' in sys.modules:
    from urllib.request import urlretrieve
    from zipfile import ZipFile
    import shutil

    cache_root = Path('/content/iml_realwaste_cache')
    dataset_root = cache_root / 'RealWaste'
    required_paths = {
        PurePosixPath(row.label) / PurePosixPath(row.filepath.replace(chr(92), '/')).name
        for frame in frames.values() for row in frame.itertuples()
    }
    missing_paths = [path for path in required_paths if not (dataset_root / str(path)).is_file()]
    if missing_paths:
        cache_root.mkdir(parents=True, exist_ok=True)
        archive_path = cache_root / 'realwaste.zip'
        download_path = cache_root / 'realwaste.zip.part'
        if not archive_path.is_file():
            print('Downloading official RealWaste dataset to runtime storage...', flush=True)
            urlretrieve('https://archive.ics.uci.edu/static/public/908/realwaste.zip', download_path)
            download_path.replace(archive_path)
        with ZipFile(archive_path) as archive:
            members = {
                PurePosixPath(*PurePosixPath(name).parts[-2:]): name
                for name in archive.namelist() if not name.endswith('/')
            }
            absent = set(missing_paths) - members.keys()
            if absent:
                raise FileNotFoundError(f'Official archive lacks manifest images: {sorted(map(str, absent))[:5]}')
            for relative_path in missing_paths:
                destination = dataset_root / str(relative_path)
                destination.parent.mkdir(parents=True, exist_ok=True)
                temporary = destination.with_suffix('.part')
                with archive.open(members[relative_path]) as source, temporary.open('wb') as target:
                    shutil.copyfileobj(source, target)
                temporary.replace(destination)
else:
    dataset_roots = [
        data_dir / 'realwaste-main' / 'RealWaste',
        project_root / 'realwaste-main' / 'RealWaste',
    ]
    dataset_root = next((root for root in dataset_roots if all((root / name).is_dir() for name in classes)), None)
    if dataset_root is None:
        raise FileNotFoundError(f'RealWaste class folders not found under {dataset_roots}')
print(f'Image source: {dataset_root}')

seen = set()
for split, frame in frames.items():
    if len(frame) != metadata[f'n_{split}'] or not frame['label'].map(metadata['label2idx']).eq(frame['label_idx']).all():
        raise ValueError(f'{split}: split size or labels do not match metadata')
    frame['relative_path'] = [
        str(PurePosixPath(label) / PurePosixPath(path.replace(chr(92), '/')).name)
        for label, path in zip(frame['label'], frame['filepath'])
    ]
    paths = set(frame['relative_path'])
    if len(paths) != len(frame) or paths & seen:
        raise ValueError(f'{split}: duplicate image paths within or across splits')
    seen.update(paths)
    missing = [path for path in paths if not (dataset_root / path).is_file()]
    if missing:
        raise FileNotFoundError(f'{split}: missing {len(missing)} images, e.g. {missing[0]}')
    for path in paths:
        with Image.open(dataset_root / path) as image:
            image.verify()
    print(split, len(frame), 'images verified')

## Record Phase 4 findings in Drive
This writes the verified progress notes into the existing project docs folder.

In [ ]:
progress_text = '# RealWaste experiment progress\n\nMobileNetV3 Large completed on a Tesla T4 using the shared seed-42 split of 3,326 training, 713 validation, and 713 test images.\n\nThe selected checkpoint was fine-tuning epoch 10, selected by validation macro F1 of 0.894055. Test accuracy was 0.872370, macro F1 was 0.876053, and cross-entropy loss was 0.555245. Training took 512.50817 seconds (about 8.5 minutes). These values were transcribed from the saved Phase 4 notebook output in Git commit d618b8d.\n\nThe classifier-head stage reached validation macro F1 of about 0.8903. Fine-tuning improved this to about 0.8941. Test accuracy improved by 13.75 percentage points over the scratch CNN baseline, and macro F1 improved by about 0.1337.\n\nThe run used ImageNet IMAGENET1K_V2 weights, six head-training epochs, ten fine-tuning epochs, and batch size 32. Images were loaded from the runtime-local official dataset cache. Manifests, checkpoints, reports, predictions, and the confusion matrix were saved under the Drive project folder by the notebook.\n\nMobileNetV3 is the only Phase 4 architecture evaluated so far. EfficientNet-B0 and GoogLeNet remain pending. Per-class findings require reading the exported report and predictions. A single seed provides a point estimate of performance.\n\nPhase 5 compares pretrained YOLOv8n classification using the same manifests. Its checkpoint selection uses validation macro F1. The final test predictions will be exported for Phase 6 comparative error analysis.\n\nReferences\n- Saved run https://github.com/Rushikesh-24-max/IML_Realwaste/blob/d618b8d/notebooks/phase4_pretrained_cnns.ipynb\n- YOLO classification https://docs.ultralytics.com/tasks/classify/\n'
(project_root / 'docs').mkdir(parents=True, exist_ok=True)
(project_root / 'docs' / 'phase4_progress.md').write_text(progress_text, encoding='utf-8')
print(project_root / 'docs' / 'phase4_progress.md')

## Arrange manifest images for YOLO classification
Use a new directory per run. Only training and validation images enter the training directory.

In [ ]:
import shutil
import tempfile
import ultralytics
from ultralytics import YOLO

if device.type != 'cuda':
    raise RuntimeError('Select a Colab GPU runtime, then rerun all cells.')
results_dir = project_root / 'results'
metrics_dir = results_dir / 'metrics'
plots_dir = results_dir / 'plots'
for directory in (metrics_dir, plots_dir):
    directory.mkdir(parents=True, exist_ok=True)
run_root = Path(tempfile.mkdtemp(prefix='iml_yolov8_', dir='/content'))
classification_root = run_root / 'dataset'
for split in ('train', 'val'):
    for name in classes:
        (classification_root / split / name).mkdir(parents=True, exist_ok=True)
    for row in frames[split].itertuples():
        source = dataset_root / row.relative_path
        destination = classification_root / split / row.relative_path
        destination.symlink_to(source)
    assert len(list((classification_root / split).rglob('*.jpg'))) == len(frames[split])
print('Ultralytics', ultralytics.__version__, 'dataset', classification_root)

## Select checkpoints using validation macro F1
The custom validator returns macro F1 as the trainer fitness value, so best.pt and early stopping use the same selection criterion as Phase 4.

In [ ]:
from copy import copy
from ultralytics.models.yolo.classify import ClassificationTrainer, ClassificationValidator

class MacroF1Validator(ClassificationValidator):
    def get_stats(self):
        stats = super().get_stats()
        targets = torch.cat(self.targets).cpu().numpy().reshape(-1)
        predictions = torch.cat(self.pred).cpu().numpy()[:, 0]
        macro_f1 = f1_score(targets, predictions, labels=label_ids, average='macro', zero_division=0)
        stats['metrics/macro_f1'] = float(macro_f1)
        stats['fitness'] = float(macro_f1)
        return stats

class MacroF1Trainer(ClassificationTrainer):
    def get_validator(self):
        self.loss_names = ['loss']
        return MacroF1Validator(self.test_loader, self.save_dir, args=copy(self.args), _callbacks=self.callbacks)

## Train YOLOv8n
Outputs go directly into a new Drive experiment directory. The split, package versions, and training arguments are recorded for reproducibility.

In [ ]:
from datetime import datetime, timezone
run_name = 'phase5_yolov8n_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
model = YOLO('yolov8n-cls.pt')
start = perf_counter()
model.train(
    trainer=MacroF1Trainer, data=str(classification_root), epochs=25, patience=7,
    imgsz=224, batch=32, device=0, workers=2, seed=seed, deterministic=True,
    optimizer='AdamW', lr0=0.001, weight_decay=0.0001,
    project=str(results_dir / 'yolo'), name=run_name,
    val=True, save=True, plots=True, cache=False,
)
train_seconds = perf_counter() - start
run_dir = Path(model.trainer.save_dir)
best_path = Path(model.trainer.best)
if not best_path.is_file():
    raise FileNotFoundError(best_path)
experiment = {
    'model': 'yolov8n-cls.pt', 'seed': seed, 'selection': 'validation macro F1',
    'ultralytics': ultralytics.__version__, 'torch': torch.__version__,
    'checkpoint': str(best_path), 'train_seconds': train_seconds,
    'split_counts': {name: len(frame) for name, frame in frames.items()},
    'training_args': vars(model.trainer.args),
}
(run_dir / 'experiment.json').write_text(json.dumps(experiment, indent=2, default=str), encoding='utf-8')
print('Best checkpoint', best_path)

## Evaluate the selected checkpoint on the test manifest
Predict in manifest order and map model labels back to the shared class indices. Export the report, predictions, confusion matrix, and summary.

In [ ]:
best = YOLO(str(best_path))
if set(best.names.values()) != set(classes):
    raise ValueError(f'Unexpected model classes: {best.names}')
test_paths = [str(dataset_root / path) for path in frames['test']['relative_path']]
predicted, confidences = [], []
for start_index in range(0, len(test_paths), 32):
    batch_paths = test_paths[start_index:start_index + 32]
    outputs = best.predict(source=batch_paths, imgsz=224, device=0, verbose=False)
    if len(outputs) != len(batch_paths):
        raise ValueError('Prediction count differs from input count.')
    for path, output in zip(batch_paths, outputs):
        if Path(output.path).name != Path(path).name:
            raise ValueError('Prediction order differs from the test manifest.')
        predicted.append(metadata['label2idx'][best.names[output.probs.top1]])
        confidences.append(float(output.probs.top1conf))
expected = frames['test']['label_idx'].tolist()
report = classification_report(expected, predicted, labels=label_ids, target_names=classes,
                               output_dict=True, zero_division=0)
pd.DataFrame(report).T.to_csv(run_dir / 'test_report.csv')
predictions = frames['test'][['relative_path', 'label', 'label_idx']].copy()
predictions['predicted_label'] = [classes[index] for index in predicted]
predictions['confidence'] = confidences
predictions.to_csv(run_dir / 'test_predictions.csv', index=False)
history = pd.read_csv(run_dir / 'results.csv')
history.columns = history.columns.str.strip()
selected = history.loc[history['metrics/macro_f1'].idxmax()]
summary = {
    'model': 'YOLOv8n classification', 'test_accuracy': accuracy_score(expected, predicted),
    'test_macro_f1': f1_score(expected, predicted, labels=label_ids, average='macro', zero_division=0),
    'val_macro_f1': float(selected['metrics/macro_f1']), 'selected_epoch': int(selected['epoch']),
    'train_seconds': train_seconds, 'checkpoint': str(best_path),
}
pd.DataFrame([summary]).to_csv(run_dir / 'summary.csv', index=False)
for source, target in [('summary.csv', 'phase5_yolov8n_summary.csv'),
                       ('test_report.csv', 'phase5_yolov8n_report.csv'),
                       ('test_predictions.csv', 'phase5_yolov8n_predictions.csv')]:
    shutil.copy2(run_dir / source, metrics_dir / target)
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(expected, predicted, labels=label_ids, display_labels=classes,
                                        xticks_rotation=45, ax=ax)
ax.set_title('YOLOv8n classification test confusion matrix')
fig.savefig(plots_dir / 'phase5_yolov8n_confusion.png', bbox_inches='tight')
plt.show()
print(json.dumps(summary, indent=2))